# Per-channel normalization strategies

This notebook confirms the behaviour of each `ChannelStrategy` fitting rule (min-max p99.9,
robust IQR, fixed division by pi, z-score) and the optional log1p pre-transform. For each
strategy we fit on a synthetic distribution, plot the raw versus normalized histogram, and
report the fitted location and scale.

Modules exercised:

- `configuration.norm_config.ChannelStrategy`
- `configuration.norm_config.NormMethod`
- `configuration.norm_config.Presets`

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy             as np
import matplotlib.pyplot as plt

import matplotlib
matplotlib.rcParams["figure.dpi"] = 110
matplotlib.rcParams["image.interpolation"] = "nearest"

SEED = 0
np.random.seed(SEED)

try:
    import torch
    torch.manual_seed(SEED)
except Exception:
    pass

print("repo root on path:", REPO_ROOT)



## Synthetic heavy-tailed distribution

SAR magnitudes are strongly right-skewed. We draw from a log-normal mixture so the difference
between min-max, IQR and z-score normalisation is visible, plus a uniform phase-like channel in
[-pi, pi] for the fixed division strategy.


In [ ]:
from configuration.norm_config import ChannelStrategy, NormMethod, Presets

rng       = np.random.default_rng(SEED)
magnitude = np.concatenate([rng.lognormal(0.0, 0.5, 8000), rng.lognormal(1.5, 0.4, 2000)]).astype(np.float32)
phase     = rng.uniform(-np.pi, np.pi, 10000).astype(np.float32)
print("magnitude range:", magnitude.min(), magnitude.max())
print("phase range    :", phase.min(), phase.max())



## Apply each strategy

We reproduce the normalisation formula used by the `Normalizer`: optional log1p, then
`(x - loc) / scale`. The fitted (loc, scale) come straight from `ChannelStrategy.fit`.


In [ ]:
def apply_strategy(strategy, data):
    loc, scale = strategy.fit(data)
    x          = np.log1p(np.maximum(data, 0.0)) if strategy.apply_log1p else data
    return (x - loc) / scale, loc, scale

magnitude_strategies = {
    "min_max_p999"        : Presets.MIN_MAX,
    "min_max_p999 +log1p" : Presets.MIN_MAX_LOG1P,
    "robust_iqr"          : Presets.ROBUST_IQR,
    "zscore"              : Presets.ZSCORE,
}

fig, axes = plt.subplots(2, len(magnitude_strategies), figsize=(3 * len(magnitude_strategies), 5))
for col, (name, strat) in enumerate(magnitude_strategies.items()):
    normed, loc, scale = apply_strategy(strat, magnitude)
    axes[0, col].hist(magnitude, bins=60, color="0.6")
    axes[0, col].set_title("raw magnitude" if col == 0 else "")
    axes[1, col].hist(normed, bins=60, color="C0")
    axes[1, col].set_title(f"{name}\nloc={loc:.3f} scale={scale:.3f}", fontsize=9)
plt.tight_layout()
plt.show()



## Fixed division by pi for phase

The phase strategy maps [-pi, pi] onto [-1, 1] with a fixed scale, independent of the data.


In [ ]:
phase_strat        = Presets.FIXED_DIV_PI
normed_phase, loc, scale = apply_strategy(phase_strat, phase)
print("fixed_div_pi loc, scale:", loc, scale)

fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].hist(phase, bins=60, color="0.6");  axes[0].set_title("raw phase [rad]")
axes[1].hist(normed_phase, bins=60, color="C2"); axes[1].set_title("phase / pi")
axes[1].axvline(-1, color="C3", ls="--"); axes[1].axvline(1, color="C3", ls="--")
plt.tight_layout()
plt.show()
print("normalized phase range:", normed_phase.min(), normed_phase.max())


## Expected visual outcome

The min-max p99.9 strategy should compress most of the magnitude mass into roughly [0, 1] while
clipping the tail beyond 1; the log1p variant should make the histogram far more symmetric;
robust IQR should centre on the median; z-score should centre at zero with unit spread. The
fixed-division phase histogram should fall exactly within [-1, 1], shown by the dashed guides.